<a href="https://colab.research.google.com/github/D4rsh11/F1-laptime-predictor/blob/main/F1_lap_time_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the F1 data library
!pip install fastf1

# Import libraries
import fastf1 as ff1
import pandas as pd
import os
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [ ]:
# Create cache directory for faster subsequent runs
cache_dir='/content/f1_cache'
os.makedirs(cache_dir, exist_ok=True)
ff1.Cache.enable_cache(cache_dir)

In [ ]:
# Get the data
ff1.Cache.enable_cache(cache_dir)
session = ff1.get_session(2024, 'Bahrain', 'R')
session.load()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.0]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core         

In [ ]:
# Check what we got
print(f"We have {len(session.laps)} laps")
print(f"Drivers:{session.laps['Driver'].unique()}")

We have 1129 laps
Drivers:['VER' 'PER' 'SAI' 'LEC' 'RUS' 'NOR' 'HAM' 'PIA' 'ALO' 'STR' 'ZHO' 'MAG'
 'RIC' 'TSU' 'ALB' 'HUL' 'OCO' 'GAS' 'BOT' 'SAR']


In [ ]:
# Clean up the data
df = session.laps.copy()

# Convert lap times to seconds
df['seconds']=df['LapTime'].dt.total_seconds()

# Remove invalid laps (pit entries/exits and formation laps)
df=df.dropna(subset=['seconds'])
print(f"After cleaning:{len(df)} laps")

After cleaning:1127 laps


In [ ]:
# Look at the data
df[['Driver', 'LapNumber', 'Stint', 'Compound', 'seconds']].head()

,Driver,LapNumber,Stint,Compound,seconds
0,VER,1.0,1.0,SOFT,97.284
1,VER,2.0,1.0,SOFT,96.296
2,VER,3.0,1.0,SOFT,96.753
3,VER,4.0,1.0,SOFT,96.647
4,VER,5.0,1.0,SOFT,97.173


In [ ]:
# Prepare for ML
data = df[['Driver', 'LapNumber', 'Stint', 'Compound', 'seconds']].copy()
data = pd.get_dummies(data, columns=['Driver', 'Compound'])

X = data.drop('seconds', axis=1)
y = data['seconds']

In [ ]:

# Train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)


GradientBoostingRegressor(random_state=42)

In [ ]:
# Test it
predictions=model.predict(X_test)
error=mean_absolute_error(y_test, predictions)
print(f"Model error:{error:} seconds")



Model error:1.890365841023263 seconds


In [ ]:
# Get list of all drivers
drivers=session.laps['Driver'].unique()
results=[]

# Predict best possible lap for each driver
for driver in drivers:
    input_data=pd.DataFrame(0, index=[0], columns=X.columns)

    input_data['LapNumber']=35
    input_data['Stint']=2

    if f'Driver_{driver}' in X.columns:
        input_data[f'Driver_{driver}']=1

    if 'Compound_SOFT' in X.columns:
        input_data['Compound_SOFT']=1

    predicted_time=model.predict(input_data)[0]
    results.append({'Driver': driver, 'Predicted': predicted_time})

In [ ]:

# Get actual fastest laps
actual_fastest = df.groupby('Driver')['seconds'].min().reset_index()
actual_fastest.columns = ['Driver', 'Actual']


In [ ]:
# Compare
comparison = pd.merge(pd.DataFrame(results), actual_fastest, on='Driver')
comparison['diff'] = comparison['Predicted'] - comparison['Actual']
comparison = comparison.sort_values('Actual')


In [ ]:
# Show results
print("\n\t\t     Predicted   vs   Actual Fastest Laps")
for i, row in comparison.iterrows():
    print(f"{row['Driver']:15} Predicted: {row['Predicted']:.2f}s \t Actual: {row['Actual']:.2f}s \t Diff: {row['diff']:+.2f}s")


		     Predicted   vs   Actual Fastest Laps
VER             Predicted: 98.35s 	 Actual: 92.61s 	 Diff: +5.74s
LEC             Predicted: 98.65s 	 Actual: 94.09s 	 Diff: +4.56s
ALO             Predicted: 98.95s 	 Actual: 94.20s 	 Diff: +4.75s
PER             Predicted: 98.74s 	 Actual: 94.36s 	 Diff: +4.37s
NOR             Predicted: 98.47s 	 Actual: 94.48s 	 Diff: +4.00s
SAI             Predicted: 100.98s 	 Actual: 94.51s 	 Diff: +6.48s
HAM             Predicted: 98.81s 	 Actual: 94.72s 	 Diff: +4.08s
SAR             Predicted: 98.31s 	 Actual: 94.73s 	 Diff: +3.57s
PIA             Predicted: 98.81s 	 Actual: 94.77s 	 Diff: +4.03s
GAS             Predicted: 97.55s 	 Actual: 94.81s 	 Diff: +2.74s
HUL             Predicted: 98.00s 	 Actual: 94.83s 	 Diff: +3.17s
RUS             Predicted: 97.05s 	 Actual: 95.06s 	 Diff: +1.98s
RIC             Predicted: 99.44s 	 Actual: 95.16s 	 Diff: +4.28s
ZHO             Predicted: 98.39s 	 Actual: 95.46s 	 Diff: +2.94s
MAG             Predicted: 98.

In [ ]:
# Overall accuracy

accuracy = (1 - mean_absolute_percentage_error(comparison['Actual'], comparison['Predicted'])) * 100
print(f"\nModel accuracy: {accuracy:.2f}%")


Model accuracy: 96.07%
